# Araseの電磁場 despun データのチェック → DSI座標系の時点で擾乱が見えるかを確認。

# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# Araseの電場・磁場データのplot

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/PSD_dsi'

psp.erg.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', get_support_data=True, no_update=True)
psp.erg.pwe_efd(trange=time_range, level='l2', datatype='E_spin', coord='dsi', get_support_data=True, no_update=True)

psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', get_support_data=True, no_update=True)
psp.erg.mgf(trange=time_range, level='l2', datatype='8sec', coord='dsi', get_support_data=True, no_update=True)

print("--- Loaded tplot variables ---")
print(pt.tplot_names())

In [ ]:
time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]

E64_data_dsi_x  = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_y  = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_quality_flag = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_quality_flag'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

import xarray as xr
import numpy as np

# QF = 0 の時間を抽出
bad_times = E64_data_dsi_quality_flag.time.where(E64_data_dsi_quality_flag != 0, drop=True)

# 実際には単純な条件で十分
E64_data_dsi_x_qf = E64_data_dsi_x.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_x.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)
E64_data_dsi_y_qf = E64_data_dsi_y.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_y.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)

ds_E64_dsi = xr.Dataset({
    'E64_dsi_x': E64_data_dsi_x_qf,
    'E64_dsi_y': E64_data_dsi_y_qf
})

ds_E64_dsi  = ds_E64_dsi.dropna(dim='time', how='all')

In [ ]:
Espin_data_dsi_u    = pt.data_quants['erg_pwe_efd_l2_E_spin_Eu_dsi'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
Espin_data_dsi_v    = pt.data_quants['erg_pwe_efd_l2_E_spin_Ev_dsi'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
Espin_data_dsi_quality_flag = pt.data_quants['erg_pwe_efd_l2_E_spin_quality_flag'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

import xarray as xr
import numpy as np

# QF = 0 の時間を抽出
bad_times = Espin_data_dsi_quality_flag.time.where(Espin_data_dsi_quality_flag != 0, drop=True)

# 実際には単純な条件で十分
Espin_data_dsi_u_qf = Espin_data_dsi_u.where(Espin_data_dsi_quality_flag.interp(time=Espin_data_dsi_u.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)
Espin_data_dsi_v_qf = Espin_data_dsi_v.where(Espin_data_dsi_quality_flag.interp(time=Espin_data_dsi_v.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)

ds_Espin_dsi_u = xr.Dataset({
    'Espin_dsi_u_x':    Espin_data_dsi_u_qf[:, 0],
    'Espin_dsi_u_y':    Espin_data_dsi_u_qf[:, 1]
})

ds_Espin_dsi_v = xr.Dataset({
    'Espin_dsi_v_x':    Espin_data_dsi_v_qf[:, 0],
    'Espin_dsi_v_y':    Espin_data_dsi_v_qf[:, 1]
})

ds_Espin_dsi_u  = ds_Espin_dsi_u.dropna(dim='time', how='all')
ds_Espin_dsi_v  = ds_Espin_dsi_v.dropna(dim='time', how='all')

In [ ]:
B64_data_dsi    = pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi_quality_flag   = pt.data_quants['erg_mgf_l2_quality_64hz'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

In [ ]:
Bspin_data_dsi  = pt.data_quants['erg_mgf_l2_mag_8sec_dsi'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
Bspin_data_dsi_quality_flag = pt.data_quants['erg_mgf_l2_quality_8sec'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, Bspin = xr.align(Bspin_data_dsi_quality_flag[:, 3], Bspin_data_dsi, join='inner')

Bspin_data_dsi_qf = xr.where(qf_B <= 21, Bspin, np.nan)

ds_Bspin_dsi  = xr.Dataset({
    'Bspin_dsi_x':  Bspin_data_dsi_qf[:, 0],
    'Bspin_dsi_y':  Bspin_data_dsi_qf[:, 1],
    'Bspin_dsi_z':  Bspin_data_dsi_qf[:, 2]
})

ds_Bspin_dsi  = ds_Bspin_dsi.dropna(dim='time', how='all')

In [ ]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [ ]:
ds_E64_dsi_segs = split_by_gap(ds_E64_dsi, gap_thr=np.timedelta64(63, 'ms'))
print(len(ds_E64_dsi_segs))

In [ ]:
ds_Espin_dsi_u_segs = split_by_gap(ds_Espin_dsi_u, gap_thr=np.timedelta64(32, 's'))
print(len(ds_Espin_dsi_u_segs))

ds_Espin_dsi_v_segs = split_by_gap(ds_Espin_dsi_v, gap_thr=np.timedelta64(32, 's'))
print(len(ds_Espin_dsi_v_segs))

In [ ]:
ds_B64_dsi_segs = split_by_gap(ds_B64_dsi, gap_thr=np.timedelta64(63, 'ms'))
print(len(ds_B64_dsi_segs))

ds_Bspin_dsi_segs   = split_by_gap(ds_Bspin_dsi, gap_thr=np.timedelta64(32, 's'))
print(len(ds_Bspin_dsi_segs))

# Wavelet analysis

In [ ]:
import os
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt
import pywt

sys.path.append("..")
import module_handmade.tdwavelet_themis as tw
importlib.reload(tw)

vars_E64    = ['E64_dsi_x', 'E64_dsi_y']
ds_E64_dsi_cwt_seg0 = tw.cwt_from_dataset(ds_E64_dsi_segs[0], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_dsi_cwt_seg1 = tw.cwt_from_dataset(ds_E64_dsi_segs[1], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_dsi_cwt_seg2 = tw.cwt_from_dataset(ds_E64_dsi_segs[2], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_dsi_cwt_seg3 = tw.cwt_from_dataset(ds_E64_dsi_segs[3], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_dsi_cwt_seg4 = tw.cwt_from_dataset(ds_E64_dsi_segs[4], dt=1/64, s0=2, dj=1/32, variables=vars_E64)

vars_Espin_u    = ['Espin_dsi_u_x', 'Espin_dsi_u_y']
ds_Espin_dsi_u_cwt_seg0 = tw.cwt_from_dataset(ds_Espin_dsi_u_segs[0], dt=8., s0=2, dj=1/32, variables=vars_Espin_u)
ds_Espin_dsi_u_cwt_seg1 = tw.cwt_from_dataset(ds_Espin_dsi_u_segs[1], dt=8., s0=2, dj=1/32, variables=vars_Espin_u)
ds_Espin_dsi_u_cwt_seg2 = tw.cwt_from_dataset(ds_Espin_dsi_u_segs[2], dt=8., s0=2, dj=1/32, variables=vars_Espin_u)
ds_Espin_dsi_u_cwt_seg3 = tw.cwt_from_dataset(ds_Espin_dsi_u_segs[3], dt=8., s0=2, dj=1/32, variables=vars_Espin_u)

vars_Espin_v    = ['Espin_dsi_v_x', 'Espin_dsi_v_y']
ds_Espin_dsi_v_cwt_seg0 = tw.cwt_from_dataset(ds_Espin_dsi_v_segs[0], dt=8., s0=2, dj=1/32, variables=vars_Espin_v)
ds_Espin_dsi_v_cwt_seg1 = tw.cwt_from_dataset(ds_Espin_dsi_v_segs[1], dt=8., s0=2, dj=1/32, variables=vars_Espin_v)
ds_Espin_dsi_v_cwt_seg2 = tw.cwt_from_dataset(ds_Espin_dsi_v_segs[2], dt=8., s0=2, dj=1/32, variables=vars_Espin_v)
ds_Espin_dsi_v_cwt_seg3 = tw.cwt_from_dataset(ds_Espin_dsi_v_segs[3], dt=8., s0=2, dj=1/32, variables=vars_Espin_v)

vars_B64    = ['B64_dsi_x', 'B64_dsi_y', 'B64_dsi_z']
ds_B64_dsi_cwt_seg0 = tw.cwt_from_dataset(ds_B64_dsi_segs[0], dt=1/64, s0=2, dj=1/32, variables=vars_B64)

vars_Bspin  = ['Bspin_dsi_x', 'Bspin_dsi_y', 'Bspin_dsi_z']
ds_Bspin_dsi_cwt_seg0   = tw.cwt_from_dataset(ds_Bspin_dsi_segs[0], dt=8., s0=2, dj=1/32, variables=vars_Bspin)

print(ds_E64_dsi_cwt_seg3)
print(ds_Espin_dsi_u_cwt_seg2)
print(ds_Espin_dsi_v_cwt_seg2)
print(ds_B64_dsi_cwt_seg0)
print(ds_Bspin_dsi_cwt_seg0)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

# ---- セグメント連結（freq合わせ）----
def concat_cwt_segments(dsets, var):
    # dsets をリストに正規化
    if isinstance(dsets, xr.Dataset):
        dsets = [dsets]
    elif isinstance(dsets, (str, bytes)):
        raise TypeError("dsets は Dataset のリストにして")

    das = []
    for ds in dsets:
        if ds is None or not isinstance(ds, xr.Dataset):
            continue
        if var in ds.data_vars:
            das.append(ds[var])

    if not das:
        return None, None

    pow_cat = xr.concat(das, dim="time").sortby("time")

    coi_name = var.replace("_cwt", "_coi")
    coi_list = []
    for ds in dsets:
        if isinstance(ds, xr.Dataset) and coi_name in ds.data_vars:
            coi_list.append(ds[coi_name])
    coi_cat = xr.concat(coi_list, dim="time").sortby("time") if coi_list else None
    return pow_cat, coi_cat

# ---- 1面描画：外でax/caxを用意する ----
def plot_cwt_on_ax(ax, da_pow, da_coi=None, t0=None, minutes=5,
                   zrange=(1e-6, 1e3), yrange=(1e-2, 4.0),
                   cmap="turbo", label_left="", unit_right=""):
    # 時間切り出し
    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        da = da_pow.sel(time=slice(t0, t1))
        coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None else None
        #ax.set_xlim(t0, t1)
    else:
        da, coi = da_pow, da_coi
    if da.time.size == 0: return None, None

    T = mdates.date2num(da.time.values)
    F = da.freq.values
    Z = da.values.astype(float)

    # COIマスク（低周波側をNaN）
    if coi is not None:
        C = coi.values[:, None]
        Z = np.where(F[None, :] < C, np.nan, Z)

    # メッシュ
    Tm = np.tile(T, (F.size, 1)).T
    Fm = np.tile(F, (T.size, 1))

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto",
                        norm=LogNorm(vmin=zrange[0], vmax=zrange[1]), cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)


    # 右側カラーバー
    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(f"{unit_right}")
    return pcm, cb


In [ ]:
dsets_E64 = [ds_E64_dsi_cwt_seg0, ds_E64_dsi_cwt_seg1, ds_E64_dsi_cwt_seg2, ds_E64_dsi_cwt_seg3, ds_E64_dsi_cwt_seg4]
targets_E64 = [
    ("E64_dsi_x_cwt", r"$E_{x}$ (DSI, 64 Hz)", "[(mV/m)$^2$/Hz]"),
    ("E64_dsi_y_cwt", r"$E_{y}$ (DSI, 64 Hz)", "[(mV/m)$^2$/Hz]")
]
joined_E64 = {}
for v, _, _ in targets_E64:
    da, coi = concat_cwt_segments(dsets_E64, v)
    da = da.sortby('freq')
    if da is not None: joined_E64[v] = (da.sortby("freq"), coi)

dsets_Espin_u   = [ds_Espin_dsi_u_cwt_seg0, ds_Espin_dsi_u_cwt_seg1, ds_Espin_dsi_u_cwt_seg2, ds_Espin_dsi_u_cwt_seg3]
targets_Espin_u = [
    ("Espin_dsi_u_x_cwt", r"$E_{x}$ (DSI, E_spin, U)", "[(mV/m)$^2$/Hz]"),
    ("Espin_dsi_u_y_cwt", r"$E_{y}$ (DSI, E_spin, U)", "[(mV/m)$^2$/Hz]")
]
joined_Espin_u  = {}
for v, _, _ in targets_Espin_u:
    da, coi = concat_cwt_segments(dsets_Espin_u, v)
    da = da.sortby('freq')
    if da is not None: joined_Espin_u[v] = (da.sortby("freq"), coi)

dsets_Espin_v   = [ds_Espin_dsi_v_cwt_seg0, ds_Espin_dsi_v_cwt_seg1, ds_Espin_dsi_v_cwt_seg2, ds_Espin_dsi_v_cwt_seg3]
targets_Espin_v = [
    ("Espin_dsi_v_x_cwt", r"$E_{x}$ (DSI, E_spin, V)", "[(mV/m)$^2$/Hz]"),
    ("Espin_dsi_v_y_cwt", r"$E_{y}$ (DSI, E_spin, V)", "[(mV/m)$^2$/Hz]")
]
joined_Espin_v  = {}
for v, _, _ in targets_Espin_v:
    da, coi = concat_cwt_segments(dsets_Espin_v, v)
    da = da.sortby('freq')
    if da is not None: joined_Espin_v[v] = (da.sortby("freq"), coi)

time_windows = [
    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
    for n in range(36)
]

for t0 in time_windows:
    fig, axes = plt.subplots(len(targets_E64) + len(targets_Espin_u) + len(targets_Espin_v), 1, figsize=(10, 12), sharex=True)
    for ax, (v, ylab, unit) in zip(axes[:len(targets_E64)], targets_E64):
        if v not in joined_E64: continue
        da, coi = joined_E64[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    for ax, (v, ylab, unit) in zip(axes[len(targets_E64):(len(targets_E64) + len(targets_Espin_u))], targets_Espin_u):
        if v not in joined_Espin_u: continue
        da, coi = joined_Espin_u[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 1./16.),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    for ax, (v, ylab, unit) in zip(axes[(len(targets_E64) + len(targets_Espin_u)):], targets_Espin_v):
        if v not in joined_Espin_v: continue
        da, coi = joined_Espin_v[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 1./16.),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    axes[-1].set_xlabel("time")
    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        t0_str = str(t0)
        fn_time = t0_str.replace(':', '').replace('T', '_')
        fig_path = os.path.join(path_base_save_plot, f'E_fields_dsi_cwt_{fn_time}.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

In [ ]:
dsets_B64   = [ds_B64_dsi_cwt_seg0]
targets_B64 = [
    ("B64_dsi_x_cwt", r"$B_{x}$ (DSI, 64 Hz)", "[(nT)$^2$/Hz]"),
    ("B64_dsi_y_cwt", r"$B_{y}$ (DSI, 64 Hz)", "[(nT)$^2$/Hz]"),
    ("B64_dsi_z_cwt", r"$B_{z}$ (DSI, 64 Hz)", "[(nT)$^2$/Hz]")
]
joined_B64  = {}
for v, _, _ in targets_B64:
    da, coi = concat_cwt_segments(dsets_B64, v)
    da = da.sortby('freq')
    if da is not None: joined_B64[v] = (da.sortby("freq"), coi)

dsets_Bspin = [ds_Bspin_dsi_cwt_seg0]
targets_Bspin = [
    ("Bspin_dsi_x_cwt", r"$B_{x}$ (DSI, 8 sec)", "[(nT)$^2$/Hz]"),
    ("Bspin_dsi_y_cwt", r"$B_{y}$ (DSI, 8 sec)", "[(nT)$^2$/Hz]"),
    ("Bspin_dsi_z_cwt", r"$B_{z}$ (DSI, 8 sec)", "[(nT)$^2$/Hz]")
]
joined_Bspin  = {}
for v, _, _ in targets_Bspin:
    da, coi = concat_cwt_segments(dsets_Bspin, v)
    da = da.sortby('freq')
    if da is not None: joined_Bspin[v] = (da.sortby("freq"), coi)

time_windows = [
    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
    for n in range(36)
]

for t0 in time_windows:
    fig, axes = plt.subplots((len(targets_B64) + len(targets_Bspin)), 1, figsize=(10, 12), sharex=True)
    for ax, (v, ylab, unit) in zip(axes[:(len(targets_B64))], targets_B64):
        if v not in joined_B64: continue
        da, coi = joined_B64[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    for ax, (v, ylab, unit) in zip(axes[(len(targets_B64)):], targets_Bspin):
        if v not in joined_Bspin: continue
        da, coi = joined_Bspin[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 1./16.),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    axes[-1].set_xlabel("time")
    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        t0_str = str(t0)
        fn_time = t0_str.replace(':', '').replace('T', '_')
        fig_path = os.path.join(path_base_save_plot, f'B_fields_dsi_cwt_{fn_time}.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

# 調査時刻におけるPSDのMedianを抽出

In [ ]:
da_E64_dsi_x_cwt        = joined_E64['E64_dsi_x_cwt']
da_E64_dsi_y_cwt        = joined_E64['E64_dsi_y_cwt']

da_Espin_dsi_u_x_cwt    = joined_Espin_u['Espin_dsi_u_x_cwt']
da_Espin_dsi_u_y_cwt    = joined_Espin_u['Espin_dsi_u_y_cwt']

da_Espin_dsi_v_x_cwt    = joined_Espin_v['Espin_dsi_v_x_cwt']
da_Espin_dsi_v_y_cwt    = joined_Espin_v['Espin_dsi_v_y_cwt']

da_B64_dsi_x_cwt        = joined_B64['B64_dsi_x_cwt']
da_B64_dsi_y_cwt        = joined_B64['B64_dsi_y_cwt']
da_B64_dsi_z_cwt        = joined_B64['B64_dsi_z_cwt']

da_Bspin_dsi_x_cwt      = joined_Bspin['Bspin_dsi_x_cwt']
da_Bspin_dsi_y_cwt      = joined_Bspin['Bspin_dsi_y_cwt']
da_Bspin_dsi_z_cwt      = joined_Bspin['Bspin_dsi_z_cwt']

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

# ---- 入力 ----
pairs = [
    ('E_64_dsi_x_cwt',      da_E64_dsi_x_cwt),
    ('E_64_dsi_y_cwt',      da_E64_dsi_y_cwt),
    ('E_spinU_dsi_x_cwt',   da_Espin_dsi_u_x_cwt),
    ('E_spinU_dsi_y_cwt',   da_Espin_dsi_u_y_cwt),
    ('E_spinV_dsi_x_cwt',   da_Espin_dsi_v_x_cwt),
    ('E_spinV_dsi_y_cwt',   da_Espin_dsi_v_y_cwt),
]
t_all_start = np.datetime64('2022-09-01T21:00:00')
t_all_end   = np.datetime64('2022-09-02T00:00:00')
step        = np.timedelta64(5, 'm')  # 5分
outdir      = path_base_save_plot  # 既存の保存先を使用

# 事前に CWT 本体のみ取り出し、timeでソート
pairs_sorted = []
for name, da in pairs:
    if isinstance(da, tuple):
        da = da[0]
    if isinstance(da, xr.DataArray):
        pairs_sorted.append((name, da.sortby('time')))

def compute_median_dict(pairs_sorted, t0, t1):
    d = {}
    for name, da in pairs_sorted:
        sub = da.sel(time=slice(t0, t1))
        if sub.sizes.get('time', 0) == 0:
            continue
        if np.iscomplexobj(sub.data):
            sub = (sub.real**2 + sub.imag**2)
        d[name] = sub.median(dim='time', skipna=True)  # (freq,)
    return d

def plot_median_dict(mdict, t0, t1, outdir=None):
    fig, ax = plt.subplots(figsize=(8, 8))
    for name, med in mdict.items():
        prefix = name.split('_')[1]  # '64', 'spinU', 'spinV'
        comp   = name.split('_')[3]  # 'x','y'
        label  = f"{prefix}-{comp}"
        ax.loglog(med['freq'], med, label=label)

    ax.minorticks_on()
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Median PSD')
    ax.set_title(f"Median {str(t0)[11:16]}–{str(t1)[11:16]}  (E: (mV/m)$^2$/Hz)")
    ax.grid(True, which='both', ls=':')
    ax.set_yticks([1E-8, 1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
    ax.legend(ncol=3)
    ax.set_xlim(1e-2, 32)
    ax.set_ylim(1e-8, 1e4)
    plt.tight_layout()

    if outdir and os.path.isdir(outdir):
        fn = f"median_bs_E_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
        fig.savefig(os.path.join(outdir, fn), dpi=300, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

# ---- 5分窓でループ ----
t_starts = np.arange(t_all_start, t_all_end, step)  # 21:00, 21:05, ..., 23:25
for t0 in t_starts:
    t1 = t0 + step
    mdict = compute_median_dict(pairs_sorted, t0, t1)
    if not mdict:  # その窓でデータ無し
        continue
    plot_median_dict(mdict, t0, t1, outdir=outdir)


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

# ---- 入力 ----
pairs = [
    ('B_64Hz_dsi_x_cwt',    da_B64_dsi_x_cwt),
    ('B_64Hz_dsi_y_cwt',    da_B64_dsi_y_cwt),
    ('B_64Hz_dsi_z_cwt',    da_B64_dsi_z_cwt),
    ('B_8sec_dsi_x_cwt',    da_Bspin_dsi_x_cwt),
    ('B_8sec_dsi_y_cwt',    da_Bspin_dsi_y_cwt),
    ('B_8sec_dsi_z_cwt',    da_Bspin_dsi_z_cwt)
]
t_all_start = np.datetime64('2022-09-01T21:00:00')
t_all_end   = np.datetime64('2022-09-02T00:00:00')
step        = np.timedelta64(5, 'm')  # 5分
outdir      = path_base_save_plot  # 既存の保存先を使用

# 事前に CWT 本体のみ取り出し、timeでソート
pairs_sorted = []
for name, da in pairs:
    if isinstance(da, tuple):
        da = da[0]
    if isinstance(da, xr.DataArray):
        pairs_sorted.append((name, da.sortby('time')))

def compute_median_dict(pairs_sorted, t0, t1):
    d = {}
    for name, da in pairs_sorted:
        sub = da.sel(time=slice(t0, t1))
        if sub.sizes.get('time', 0) == 0:
            continue
        if np.iscomplexobj(sub.data):
            sub = (sub.real**2 + sub.imag**2)
        d[name] = sub.median(dim='time', skipna=True)  # (freq,)
    return d

def plot_median_dict(mdict, t0, t1, outdir=None):
    fig, ax = plt.subplots(figsize=(8, 8))
    for name, med in mdict.items():
        prefix = name.split('_')[1]  # '64', 'spinU', 'spinV'
        comp   = name.split('_')[3]  # 'x','y'
        label  = f"{prefix}-{comp}"
        ax.loglog(med['freq'], med, label=label)

    ax.minorticks_on()
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Median PSD')
    ax.set_title(f"Median {str(t0)[11:16]}–{str(t1)[11:16]}  (B: (nT)$^2$/Hz)")
    ax.grid(True, which='both', ls=':')
    ax.set_yticks([1E-8, 1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
    ax.legend(ncol=2)
    ax.set_xlim(1e-2, 32)
    ax.set_ylim(1e-8, 1e4)
    plt.tight_layout()

    if outdir and os.path.isdir(outdir):
        fn = f"median_bs_B_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
        fig.savefig(os.path.join(outdir, fn), dpi=300, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

# ---- 5分窓でループ ----
t_starts = np.arange(t_all_start, t_all_end, step)  # 21:00, 21:05, ..., 23:25
for t0 in t_starts:
    t1 = t0 + step
    mdict = compute_median_dict(pairs_sorted, t0, t1)
    if not mdict:  # その窓でデータ無し
        continue
    plot_median_dict(mdict, t0, t1, outdir=outdir)
